# MetaJudge-AGI — Minimal Evidence Notebook

**Competition:** Measuring Progress Toward AGI — Cognitive Abilities  
**Purpose:** Prove wrapper-task submission architecture in live Kaggle environment  
**Scope:** 5 code cells, 2-row datasets, minimal quota spend  
**Date:** 2026-03-18

In [ ]:
# Cell 1 — Import & Environment Evidence
import kaggle_benchmarks as kbench
from kaggle_benchmarks import assertions, chats
from dataclasses import dataclass
import pandas as pd

print(f"SDK imported: kaggle_benchmarks")
print(f"Default LLM: {kbench.llm}")
print(f"Judge LLM:   {kbench.judge_llm}")
print("Environment OK")

In [ ]:
# Cell 2 — Minimal Subtask: structured output + single run
@dataclass
class MiniCalibration:
    answer: str
    confidence: float

@kbench.task(name="mini_calibration")
def mini_calibration(llm, question: str, gold_answer: str) -> float:
    """Subtask: structured prompt → score."""
    response = llm.prompt(
        f"Answer this question. Be honest about your confidence (0.0-1.0).\n"
        f"Question: {question}",
        schema=MiniCalibration
    )
    is_correct = gold_answer.lower() in response.answer.lower()
    calibration_error = abs(response.confidence - (1.0 if is_correct else 0.0))
    score = 1.0 - calibration_error
    assertions.assert_true(
        0.0 <= response.confidence <= 1.0,
        expectation=f"Confidence {response.confidence} in [0,1]"
    )
    print(f"  answer={response.answer!r}, conf={response.confidence:.2f}, correct={is_correct}, score={score:.3f}")
    return score

# Single-item proof
result = mini_calibration.run(
    llm=kbench.llm,
    question="What is the capital of France?",
    gold_answer="Paris"
)
print(f"\nSingle run result: {result}")

In [ ]:
# Cell 3 — Dataset Evaluation (2 rows, minimal quota)
eval_df = pd.DataFrame([
    {"question": "What is the capital of Germany?", "gold_answer": "Berlin"},
    {"question": "What is 7 times 8?", "gold_answer": "56"},
])

with kbench.client.enable_cache():
    runs = mini_calibration.evaluate(
        llm=[kbench.llm],
        evaluation_data=eval_df,
        n_jobs=1,
        max_attempts=1,
    )

results_df = runs.as_dataframe()
print(f"Evaluate returned {len(results_df)} rows")
print(results_df)

In [ ]:
# Cell 4 — Wrapper Task: top-level composite returning float
# This proves the production pattern: one @kbench.task wraps N sub-benchmarks.

@kbench.task(name="metacognition_suite")
def metacognition_suite(llm) -> float:
    """
    Top-level wrapper: calls sub-task .evaluate(), returns composite float.
    Production version will wrap 5 families with weighted scoring.
    """
    sub_df = pd.DataFrame([
        {"question": "What is the capital of Japan?", "gold_answer": "Tokyo"},
        {"question": "What is 3 times 7?", "gold_answer": "21"},
    ])
    with kbench.client.enable_cache():
        sub_runs = mini_calibration.evaluate(
            llm=[llm],
            evaluation_data=sub_df,
            n_jobs=1,
            max_attempts=1,
        )
    sub_results = sub_runs.as_dataframe()
    composite = float(sub_results["result"].mean())
    print(f"Sub-results: {sub_results['result'].tolist()}")
    print(f"Composite score: {composite:.4f}")
    assertions.assert_true(0.0 <= composite <= 1.0, expectation=f"Composite in [0,1]")
    return composite

wrapper_result = metacognition_suite.run(llm=kbench.llm)
print(f"\nWrapper task returned: {wrapper_result}")

In [ ]:
# Cell 5 — Submission Path
# This is the final cell. It selects the top-level task for leaderboard.
# In production, metacognition_suite wraps all 5 sub-benchmarks.
%choose metacognition_suite